## **<font color="blue">Prebuilt Plugins</font>**
ADK includes several plugins that you can add to your agent workflow immediately:

### **<font color="green">BigQuery Agent Analytics Plugin for ADK</font>**
- The BigQuery Agent Analytics Plugin significantly enhances the Agent Development Kit (ADK) providing a robust solution for in-depth agent behavior analysis.
- Using the ADK Plugin architecture and the **BigQuery Storage Write API**, it captures and logs critical operational events directly into a Google BigQuery table empowering you with advanced capabilitiees for debugging, real-time monitoring, and comprehensive offline performance evaluation.
- Version 1.26.0 adds **Auto Schema Upgrade** (safely add new columns to existing tables), **Tool Provenance** tracking (LOCAL, MCP, SUB_AGENT, A2A, TRANSFER_AGENT), and **HITL Event Tracking** for human-in-the-loop interactions.
#### **Use Cases**
- **Agent Workflow Debugging and Analysis:** Capture a wide range of *plugin lifecycle events* (LLM calls, tool usage) and *agent-yielded events* (user input, model responses), into a well-defined schema.
- **High-Volume Analysis and Debugging:** Logging operations are performed asynchronously using the Storage Write API to allow high throughput and low latency.
- **Multimodal Analysis:** Log and analyze text, images, and other modalities. Large files are offloaded to GCS, making them accessible to BigQuery ML via Object Tables.
- **Distributed Tracking:** Built-in support for OpenTelemetry-style tracing (`trace_id`, `span_id`) to visualize agent execution flows.
- **Tool Provenance:** Track the origin of each tool call(local function, MCP server, sub-agent, A2A remote agent, or transfer agent).
- **Human-in-the-Loop (HITL) Tracking:** Dedicated event types for credential requests, confirmation prompts, and user input requests.

In [4]:
import os
import google.auth
import asyncio
from datetime import datetime
from typing import Any
import random
from google.adk.apps import App
from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools.bigquery import BigQueryToolset, BigQueryCredentialsConfig
from google.adk.plugins import ReflectAndRetryToolPlugin
from google.adk.plugins.reflect_retry_tool_plugin import TrackingScope
from google.adk.tools import FunctionTool
import google.genai.types as types
# Import the config object
from config import config
# -----------------------------
# OpenTelemetry Setup
# -----------------------------
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
current_provider = trace.get_tracer_provider()
if not isinstance(current_provider, TracerProvider):
    trace.set_tracer_provider(TracerProvider())
# -----------------------------
# Environment / GCP Config
# -----------------------------
PROJECT_ID = config.BQ_PROJECT_ID or "your-gcp-project-id"
DATASET_ID = config.BQ_DATASET or "your-big-query-dataset-id"
LOCATION = os.getenv("GOOGLE_CLOUD_LOCATION", "US")
GCS_BUCKET = config.BUCKET_NAME or "your-gcs-bucket-name"
if not PROJECT_ID:
    raise ValueError("BQ_PROJECT_ID must be set in config.py")
if not DATASET_ID:
    raise ValueError("BQ_DATASET must be set in config.py")
if not GCS_BUCKET:
    raise ValueError("BUCKET_NAME must be set in config.py")
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"  # Set to False to use API key and avoid Vertex AI permission issues
# -----------------------------
# Define Tools
# -----------------------------
def get_current_datetime() -> dict:
    if random.random() < 0.3: # 30% chance to fail
        return {"status": "error", "message": "Simulated failure"}
    now = datetime.now()
    return {
        "status": "success",
        "date": now.strftime("%d/%m/%Y"),
        "time": now.strftime("%H:%M:%S"),
        "day": now.strftime("%A")
    }
get_current_datetime_tool = FunctionTool(func=get_current_datetime)
def guess_number_tool(query: int) -> dict[str, Any]:
    target_number = 3
    if query == target_number:
        return {"status": "success", "result": "Number is valid."}
    if abs(query - target_number) <= 2:
        return {"status": "error", "error_message": "Number is almost valid."}
    if query > target_number:
        raise ValueError("Number is too large.")
    if query < target_number:
        raise ValueError("Number is too small.")
    raise ValueError("Number is invalid.")
guess_number_tool = FunctionTool(func=guess_number_tool)
# -----------------------------
# Initialize Gemini Model
# -----------------------------
llm = Gemini(model="gemini-2.5-flash", api_key=config.GOOGLE_API_KEY)
# -----------------------------
# Create LlmAgent with Tools
# -----------------------------
root_agent = LlmAgent(
    model=llm,
    name="ChatAgent",
    instruction="""
    You are a helpful assistant. Respond politely and concisely to user questions.
    Use available tools when needed.
    """,
    tools=[get_current_datetime_tool, guess_number_tool]
)
# -----------------------------
# Custom Retry Plugin
# -----------------------------
class CustomRetryPlugin(ReflectAndRetryToolPlugin):
    async def extract_error_from_result(self, *, tool, tool_args, tool_context, result):
        if isinstance(result, dict) and result.get("status") == "error":
            return result
        return None
error_handling_plugin = CustomRetryPlugin(
    max_retries=5,
    throw_exception_if_retry_exceeded=False,
    tracking_scope=TrackingScope.GLOBAL
)
# -----------------------------
# BigQuery Toolset with Explicit Credentials
# -----------------------------
from google.oauth2 import service_account
credentials = service_account.Credentials.from_service_account_file(
    config.GOOGLE_APPLICATION_CREDENTIALS,
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
bigquery_toolset = BigQueryToolset(
    credentials_config=BigQueryCredentialsConfig(credentials=credentials)
)
root_agent.tools.append(bigquery_toolset)
# -----------------------------
# Create App with Agent + Plugins (Omit BigQuery Logging Plugin to Avoid Permission Errors)
# -----------------------------
app = App(
    name="my_bq_agent",
    root_agent=root_agent,
    plugins=[error_handling_plugin]  # Removed bq_logging_plugin
)
# -----------------------------
# Session and Runner
# -----------------------------
from google.adk.sessions import InMemorySessionService
from google.adk.artifacts import InMemoryArtifactService
from google.adk.runners import Runner
APP_NAME = "chatbot_demo"
USER_ID = "user_001"
SESSION_ID = "session_001"
session_service = InMemorySessionService()
artifact_service = InMemoryArtifactService()
runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service,
    artifact_service=artifact_service,
    plugins=[error_handling_plugin]  # Removed bq_logging_plugin
)
async def create_session():
    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=SESSION_ID
    )
# -----------------------------
# Chat function
# -----------------------------
async def chat(user_input: str):
    print(f"Human: {user_input}")
    content = types.Content(
        role="user",
        parts=[types.Part(text=user_input)]
    )
   
    events = runner.run(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=content
    )
   
    for event in events:
        if event.is_final_response() and event.content and event.content.parts:
            print(f"AI: {event.content.parts[0].text}\n")
# -----------------------------
# Main conversation flow
# -----------------------------
async def main():
    await create_session()
    await chat("Hello! How are you today?")
    await chat("Can you suggest a fun weekend activity?")
    await chat("Please run get_current_datetime for me.")
    await chat("Give me a short summary of Artificial Intelligence.")
    await chat("I want to guess the number 2. Can you check if it's correct?")
    await chat("Run a BigQuery query to describe the schema of the dataset market_intelligence.agent_events.")
    await chat("Using BigQuery, count the number of rows in the table market_intelligence.agent_events.")
# Run in Jupyter Lab
await main()

C:\Users\DELL\AppData\Local\Temp\ipykernel_11448\3544799864.py:91: UserWarning: [EXPERIMENTAL] ReflectAndRetryToolPlugin: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  error_handling_plugin = CustomRetryPlugin(


Human: Hello! How are you today?
AI: Hello! I'm functioning well and ready to assist you. How can I help you today?

Human: Can you suggest a fun weekend activity?
AI: I'd love to help with that! To give you the best suggestion, could you tell me a little about what you consider "fun"? For example, do you prefer:

*   **Relaxing activities** like reading, watching movies, or getting a massage?
*   **Active outings** like hiking, biking, or playing sports?
*   **Creative pursuits** like painting, writing, or cooking?
*   **Social events** like parties, concerts, or trying new restaurants?
*   **Learning something new** like a workshop or a museum visit?

Once I have a better idea of your preferences, I can give you a more tailored recommendation! 😊


Human: Please run get_current_datetime for me.
AI: The current date is Friday, February 27, 2026, and the time is 14:46:41.

Human: Give me a short summary of Artificial Intelligence.
AI: Artificial Intelligence (AI) is a broad field of com

In [10]:
import os
import asyncio
from datetime import datetime
from typing import Any
import random

from google.adk.apps import App
from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools.bigquery import BigQueryToolset, BigQueryCredentialsConfig
from google.adk.plugins import ReflectAndRetryToolPlugin
from google.adk.plugins.reflect_retry_tool_plugin import TrackingScope
from google.adk.tools import FunctionTool

from google.oauth2 import service_account
import google.genai.types as types

# Import config (ONLY source of env values)
from config import config

# -----------------------------
# Load Configuration from config.py
# -----------------------------
PROJECT_ID = config.BQ_PROJECT_ID
DATASET_ID = config.BQ_DATASET
LOCATION = os.getenv("GOOGLE_CLOUD_LOCATION", "US")  # optional fallback

if not PROJECT_ID:
    raise ValueError("BQ_PROJECT_ID must be set in config.py")

if not DATASET_ID:
    raise ValueError("BQ_DATASET must be set in config.py")

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"

# -----------------------------
# Custom Function Tool
# -----------------------------
def get_current_datetime() -> dict:
    if random.random() < 0.3:
        return {"status": "error", "message": "Simulated failure"}
    now = datetime.now()
    return {
        "status": "success",
        "date": now.strftime("%d/%m/%Y"),
        "time": now.strftime("%H:%M:%S"),
        "day": now.strftime("%A")
    }

get_current_datetime_tool = FunctionTool(func=get_current_datetime)

# -----------------------------
# Initialize Gemini using config
# -----------------------------
llm = Gemini(
    model="gemini-2.5-flash",
    api_key=config.GOOGLE_API_KEY
)

# -----------------------------
# BigQuery Credentials using config
# -----------------------------
credentials = service_account.Credentials.from_service_account_file(
    config.GOOGLE_APPLICATION_CREDENTIALS,
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)

bigquery_toolset = BigQueryToolset(
    credentials_config=BigQueryCredentialsConfig(credentials=credentials)
)

# -----------------------------
# Root Agent
# -----------------------------
root_agent = LlmAgent(
    model=llm,
    name="ChatAgent",
    instruction=f"""
You are an AI assistant specialized in AI, ML, and GenAI.

For BigQuery:
- Always use fully qualified table names.
- The active table is:
  `{PROJECT_ID}.{DATASET_ID}.bq_plugin`

Schema:
id INT64
customer_name STRING
product STRING
quantity INT64
price FLOAT64
sale_date DATE
created_at TIMESTAMP
""",
    tools=[get_current_datetime_tool, bigquery_toolset]
)

# -----------------------------
# Retry Plugin
# -----------------------------
class CustomRetryPlugin(ReflectAndRetryToolPlugin):
    async def extract_error_from_result(self, *, tool, tool_args, tool_context, result):
        if isinstance(result, dict) and result.get("status") == "error":
            return result
        return None

error_plugin = CustomRetryPlugin(
    max_retries=5,
    throw_exception_if_retry_exceeded=False,
    tracking_scope=TrackingScope.GLOBAL
)

# -----------------------------
# Session & Runner
# -----------------------------
from google.adk.sessions import InMemorySessionService
from google.adk.artifacts import InMemoryArtifactService
from google.adk.runners import Runner

APP_NAME = "bq_plugin_demo"
USER_ID = "user_001"
SESSION_ID = "session_001"

session_service = InMemorySessionService()
artifact_service = InMemoryArtifactService()

runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service,
    artifact_service=artifact_service,
    plugins=[error_plugin]
)

# -----------------------------
# Create Session
# -----------------------------
async def create_session():
    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=SESSION_ID
    )

# -----------------------------
# Chat Function
# -----------------------------
async def chat(user_input: str):
    print(f"\nHuman: {user_input}")

    content = types.Content(
        role="user",
        parts=[types.Part(text=user_input)]
    )

    events = runner.run(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=content
    )

    for event in events:
        if event.is_final_response() and event.content and event.content.parts:
            print(f"AI: {event.content.parts[0].text}")

# -----------------------------
# Main
# -----------------------------
async def main():
    await create_session()

    await chat("Describe the schema of the bq_query table.")
    await chat("Count the total rows.")
    await chat("Calculate total revenue grouped by product.")

await main()

C:\Users\DELL\AppData\Local\Temp\ipykernel_11448\2257981492.py:109: UserWarning: [EXPERIMENTAL] ReflectAndRetryToolPlugin: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  error_plugin = CustomRetryPlugin(



Human: Describe the schema of the bq_query table.
AI: The `bq_plugin` table has the following schema:
*   **id**: INTEGER
*   **customer_name**: STRING
*   **product**: STRING
*   **quantity**: INTEGER
*   **price**: FLOAT
*   **sale_date**: DATE
*   **created_at**: TIMESTAMP (with a default value expression of `CURRENT_TIMESTAMP()`)

Human: Count the total rows.
AI: The `bq_plugin` table has 3 rows.

Human: Calculate total revenue grouped by product.
AI: Here is the total revenue grouped by product:

*   **Tablet**: 1350.75
*   **Laptop**: 1200.5
*   **Phone**: 1600.0
